# Day 6 — Data Cleaning
Internship Task 6 | 23 November 2026

A deliberately messy copy of the original Day 5 retail sales dataset is used. Keep both CSV files in the notebook folder.

## 1. Load dataset and inspect

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("Day06_Before_Cleaning.csv",dtype=str)
print("Shape before:",df.shape)
print(df.head().to_string(index=False))

Shape before: (32, 10)
Order_ID Order_Date Customer_Name   City   Category    Product Quantity Unit_Price Sales Customer_Rating
  ORD001 2026-10-02         Rahul  pune    Clothing    T-Shirt        2        650  1300               5
  ORD002 2026-10-03          Neha Nagpur    grocery       Rice        3        320   960               3
  ORD003 2026-10-04        Vikram Nashik       HOME       Lamp        4       1100  4400               4
  ORD004 2026-10-05         Priya   Pune Electronic Headphones        5       1500  7500               5
  ORD005 2026-10-06         Arjun MUMBAI   Clothing    T-Shirt        1        650   650               4


## 2. Identify missing values

In [2]:
df = df.replace(r"^\s*$",np.nan,regex=True)
print(df.isnull().sum().to_string())

Order_ID           0
Order_Date         0
Customer_Name      1
City               1
Category           0
Product            0
Quantity           0
Unit_Price         0
Sales              1
Customer_Rating    3


## 3. Count exact duplicates and duplicate order IDs

In [3]:
print("Exact duplicates:",df.duplicated().sum())
print("Duplicate Order_IDs:",df.Order_ID.duplicated().sum())
print(df[df.Order_ID.duplicated(keep=False)][["Order_ID","Sales"]].to_string(index=False))

Exact duplicates: 2
Duplicate Order_IDs: 3
Order_ID Sales
  ORD001  1300
  ORD002   960
  ORD018  1280
  ORD018  5500
  ORD001  1300
  ORD002   960


## 4. Fix types and standardize text

In [4]:
clean=df.drop_duplicates().copy()
for col in ["Order_ID","Customer_Name","City","Category","Product"]:
    clean[col]=clean[col].astype("string").str.strip()
for col in ["Customer_Name","City","Product","Category"]:
    clean[col]=clean[col].str.title()
clean["Category"]=clean["Category"].replace({"Electronic":"Electronics"})
for col in ["Quantity","Unit_Price","Sales","Customer_Rating"]:
    clean[col]=pd.to_numeric(clean[col],errors="coerce")
clean["Order_Date"]=pd.to_datetime(clean["Order_Date"],errors="coerce",format="mixed")
print(clean.dtypes.to_string())

Order_ID           string[python]
Order_Date         datetime64[ns]
Customer_Name      string[python]
City               string[python]
Category           string[python]
Product            string[python]
Quantity                  float64
Unit_Price                float64
Sales                     float64
Customer_Rating           float64


## 5. Handle missing, invalid values and duplicates
Business rules: quantity and price must be positive, ratings between 1 and 5. Remove records with missing essential fields; fill optional names and cities with Unknown; impute missing rating with median. Conflicting duplicate IDs are flagged, and the first is retained for this exercise only.

In [5]:
clean.loc[clean.Quantity<=0,"Quantity"]=np.nan
clean.loc[clean.Unit_Price<=0,"Unit_Price"]=np.nan
clean.loc[~clean.Customer_Rating.between(1,5),"Customer_Rating"]=np.nan
clean["Customer_Name"]=clean.Customer_Name.fillna("Unknown")
clean["City"]=clean.City.fillna("Unknown")
clean["Customer_Rating"]=clean.Customer_Rating.fillna(clean.Customer_Rating.median())
print("Conflicting duplicate IDs:",clean.Order_ID.duplicated().sum())
clean=clean.drop_duplicates(subset="Order_ID",keep="first")
clean=clean.dropna(subset=["Order_ID","Order_Date","Quantity","Unit_Price","Category","Product"]).copy()
clean["Quantity"]=clean.Quantity.astype(int)
clean["Sales"]=(clean.Quantity*clean.Unit_Price).round(2)
print("Missing after cleaning:\n",clean.isnull().sum().to_string())

Conflicting duplicate IDs: 1
Missing after cleaning:
 Order_ID           0
Order_Date         0
Customer_Name      0
City               0
Category           0
Product            0
Quantity           0
Unit_Price         0
Sales              0
Customer_Rating    0


## 6. Identify abnormal sales values with IQR
IQR flags potential outliers for review; do not delete legitimate large orders automatically.

In [6]:
q1,q3=clean.Sales.quantile([.25,.75]);iqr=q3-q1
lo,hi=q1-1.5*iqr,q3+1.5*iqr
outliers=clean[(clean.Sales<lo)|(clean.Sales>hi)]
print("IQR thresholds:",round(lo,2),round(hi,2))
print("Potential outliers:",len(outliers))
print(outliers[["Order_ID","Sales"]].to_string(index=False))

IQR thresholds: -3375.0 8185.0
Potential outliers: 0
Empty DataFrame
Columns: [Order_ID, Sales]
Index: []


## 7. Export before/after datasets and verify

In [7]:
clean=clean.sort_values("Order_ID").reset_index(drop=True)
clean["Order_Date"]=clean.Order_Date.dt.strftime("%Y-%m-%d")
clean.to_csv("Day06_After_Cleaning.csv",index=False)
print("Before:",len(df),"After:",len(clean))
print("Exact duplicates after:",clean.duplicated().sum())
print("Duplicate IDs after:",clean.Order_ID.duplicated().sum())
print("Cleaned total sales:",round(clean.Sales.sum(),2))
assert clean.Order_ID.is_unique and clean.Quantity.gt(0).all() and clean.Unit_Price.gt(0).all()
assert np.allclose(clean.Sales,clean.Quantity*clean.Unit_Price)
print("Validation passed")

Before: 32 After: 23
Exact duplicates after: 0
Duplicate IDs after: 0
Cleaned total sales: 61000.0
Validation passed


## Screenshot / Proof Checklist
Capture VS Code screenshots of: missing counts; duplicate counts; corrected types; outlier check; final validation.

Methods: `isnull()`, `fillna()`, `dropna()`, `duplicated()`, `drop_duplicates()`, `astype()`, `replace()`, `to_csv()`.